# Exploratory Data Analysis (EDA): Real Music Representations & Structure Graphs
### Course: Neural Networks (CSE425 / EEE474 / CSE715)
**Project**: GNN-Based BERT for Understanding Context from Music

This notebook explores the multi-layered music signal using **real audio files** from MagnaTagATune and MusicCaps:
1. **Real Acoustic Signals**: Waveform, 128-bin Log-Mel Spectrogram, and 12-bin Pitch Chroma.
2. **Graph Construction**: Transforming audio segments into graph nodes $V$ connected by temporal adjacency and harmonic/acoustic similarity edges $E$ (cosine similarity $\tau > 0.55$).
3. **Contextual Metadata**: Multi-label tag distributions from real annotations.

In [ ]:
import os
import sys
import json
import numpy as np
import matplotlib.pyplot as plt
import networkx as nx

# Add project root to sys.path
sys.path.insert(0, os.path.abspath('..'))

from src.audio_features import (
    load_and_resample_audio,
    extract_log_mel_spectrogram,
    extract_chroma_features,
    extract_segment_features,
)
from src.graph_builder import build_segment_graph, to_networkx_graph

print("EDA libraries successfully imported!")

## 1. Real Audio Signal Inspection: Mel-Spectrogram & Chroma
Load and inspect a real recording from `/Users/tanishaislam/MusicCaps/audio/mc_001.wav`.

In [ ]:
sr = 22050
audio_path = "/Users/tanishaislam/MusicCaps/audio/mc_001.wav"
if not os.path.exists(audio_path):
    audio_path = "/Users/tanishaislam/MusicCaps/MagnaTagATune/audio/aba_structure-epic-01-deep_step-117-146.mp3"

audio = load_and_resample_audio(audio_path, target_sr=sr, duration=10.0)
log_mel = extract_log_mel_spectrogram(audio, sr=sr, n_mels=128)
chroma = extract_chroma_features(audio, sr=sr, n_chroma=12)

duration = len(audio) / sr
fig, axes = plt.subplots(3, 1, figsize=(12, 9), sharex=False)

# 1. Waveform
t = np.linspace(0, duration, len(audio))
axes[0].plot(t, audio, color='#2563eb', alpha=0.8)
axes[0].set_title(f'Real Audio Waveform (22,050 Hz) - {os.path.basename(audio_path)}', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Amplitude')
axes[0].set_xlim(0, duration)
axes[0].grid(True, alpha=0.3)

# 2. Log-Mel Spectrogram
im1 = axes[1].imshow(log_mel, aspect='auto', origin='lower', cmap='magma')
axes[1].set_title('Normalized 128-bin Log-Mel Spectrogram', fontsize=12, fontweight='bold')
axes[1].set_ylabel('Mel Frequency Bins')
fig.colorbar(im1, ax=axes[1], fraction=0.02, pad=0.01)

# 3. 12-Pitch Chroma
im2 = axes[2].imshow(chroma, aspect='auto', origin='lower', cmap='viridis')
axes[2].set_title('12-bin Pitch Chroma Features (Harmonic Signature)', fontsize=12, fontweight='bold')
axes[2].set_ylabel('Pitch Class (C..B)')
axes[2].set_xlabel('Time Frames')
fig.colorbar(im2, ax=axes[2], fraction=0.02, pad=0.01)

plt.tight_layout()
plt.show()

## 2. Music Segment Graph Construction from Real Audio
Segments the real audio track into fixed temporal windows (2.5s), computes node features $h_i^{(0)} \in \mathbb{R}^{32}$, and establishes edges via:
- **Temporal Adjacency**: $i \leftrightarrow i+1$
- **Harmonic Cosine Similarity**: $\cos(h_i, h_j) > \tau$ (threshold $\tau = 0.55$)

In [ ]:
segment_feats = extract_segment_features(audio, sr=sr, segment_duration=2.5)
graph_dict = build_segment_graph(segment_feats, similarity_threshold=0.55)
G = to_networkx_graph(graph_dict)

print(f"Constructed Real Audio Graph: {G.number_of_nodes()} Nodes, {G.number_of_edges()} Edges")

plt.figure(figsize=(8, 6))
pos = nx.spring_layout(G, seed=42)
nx.draw_networkx_nodes(G, pos, node_size=700, node_color='#3b82f6', alpha=0.9)
nx.draw_networkx_labels(G, pos, font_color='white', font_weight='bold')
nx.draw_networkx_edges(G, pos, edge_color='#64748b', width=2, alpha=0.8)
edge_labels = nx.get_edge_attributes(G, 'weight')
nx.draw_networkx_edge_labels(G, pos, edge_labels=edge_labels, font_size=8)

plt.title('Real Music Structure Graph G = (V, E) with Acoustic Similarity Weights', fontsize=12, fontweight='bold')
plt.axis('off')
plt.show()

## 3. Real MagnaTagATune Multi-Label Tag Distribution

In [ ]:
with open('../data/processed/metadata_catalog.json', 'r') as f:
    catalog = json.load(f)

all_tags = []
for item in catalog:
    all_tags.extend(item.get('tags', []))

from collections import Counter
counts = Counter(all_tags).most_common(16)
tag_names = [c[0] for c in counts]
tag_freqs = [c[1] for c in counts]

plt.figure(figsize=(10, 5))
plt.barh(tag_names[::-1], tag_freqs[::-1], color='#3b82f6')
plt.title('Top Ground-Truth Tag Frequencies Across Real Processed Tracks', fontsize=12, fontweight='bold')
plt.xlabel('Frequency Count')
plt.grid(True, linestyle=':', alpha=0.5)
plt.tight_layout()
plt.show()